In [1]:
%pip -q install pandas numpy openai tqdm

You should consider upgrading via the '/home/ec2-user/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from openai import OpenAI

client = OpenAI()  # uses OPENAI_API_KEY from env

models = client.models.list()
for m in models.data:
    print(m.id)


gpt-4-0613
gpt-4
gpt-3.5-turbo
gpt-5.2-codex
gpt-4o-mini-tts-2025-12-15
gpt-realtime-mini-2025-12-15
gpt-audio-mini-2025-12-15
chatgpt-image-latest
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
dall-e-3
dall-e-2
gpt-4-1106-preview
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-4-0125-preview
gpt-4-turbo-preview
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
chatgpt-4o-latest
gpt-4o-audio-preview
gpt-4o-realtime-preview
omni-moderation-latest
omni-moderation-2024-09-26
gpt-4o-realtime-preview-2024-12-17
gpt-4o-audio-preview-2024-12-17
gpt-4o-mini-realtime-preview-2024-12-17
gpt-4o-mini-audio-preview-2024-12-17
o1-2024-12-17
o1
gpt-4o-mini-realtime-preview
gpt-4o-mini-audio-preview
o3-mini
o3-mini-2025-01-31
gpt-4o-2024-11-20
gpt-4o-search-preview-2025-03-11
gpt-4o-search-preview
gpt-4o-mini-search-preview-2025-03-11
gpt

In [3]:
import os
import re
import time
import pandas as pd
import numpy as np
from tqdm import tqdm

# ---------- Choose ONE provider ----------

# Option A) OpenAI (recommended if you have OPENAI_API_KEY set)
from openai import OpenAI
client = OpenAI()  # uses env var OPENAI_API_KEY

# Option B) HuggingFace Router / other OpenAI-compatible endpoint:
# from openai import OpenAI
# client = OpenAI(
#     base_url="https://router.huggingface.co/v1",
#     api_key=os.environ["HF_TOKEN"],  # set HF_TOKEN in env
# )

MODEL = "gpt-5.2"     # or "gpt-5.2-mini" if you want cheaper/faster
TEMPERATURE = 0
MAX_COMPLETION_TOKENS = 80


In [4]:
CSV_PATH = "Ambivalent_Binning.csv"
df = pd.read_csv(CSV_PATH, dtype=str)  # keep everything as string-safe

print("Rows:", len(df))
print("Columns:", len(df.columns))

bin_cols = [c for c in df.columns if c.startswith("bin") and c.endswith("_comment")]
print("Bin columns:", len(bin_cols), "->", bin_cols[:5], "...")

df.head(2)


Rows: 1082
Columns: 34
Bin columns: 22 -> ['bin01_comment', 'bin02_comment', 'bin03_comment', 'bin04_comment', 'bin05_comment'] ...


,post_id,subreddit,title,author,score,num_comments_listed,op_replied,op_reply_count,created_utc,permalink,...,bin13_comment,bin14_comment,bin15_comment,bin16_comment,bin17_comment,bin18_comment,bin19_comment,bin20_comment,bin21_comment,bin22_comment
0,1hv4zdy,meToo,Was this SA?,Ok-Sugar959,5.0,7.0,True,3,1736185759.0,/r/meToo/comments/1hv4zdy/was_this_sa/,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1f2kibx,meToo,Dealing with sexual assault trauma / was it SA?,Ok-Air-6389,6.0,3.0,True,1,1724773946.0,/r/meToo/comments/1f2kibx/dealing_with_sexual_...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
import re

# Automatically detect all bin columns in order: bin01_comment ... bin22_comment (or more)
BIN_COLS = sorted(
    [c for c in df.columns if re.match(r"^bin\d+_comment$", c)],
    key=lambda x: int(re.findall(r"\d+", x)[0])
)

def get_story_text(row: pd.Series) -> str:
    # Prefer combined_text, else body, else title
    for col in ["combined_text", "body", "title"]:
        if col in row and pd.notna(row[col]):
            s = str(row[col]).strip()
            if s and s.lower() != "nan":
                return s
    return ""

def get_comments_list(row: pd.Series):
    comments = []
    for col in BIN_COLS:
        if col in row and pd.notna(row[col]):
            s = str(row[col]).strip()
            if s and s.lower() != "nan":
                comments.append(s)
    return comments


In [6]:
PROMPT_TEMPLATE = """

I will give you 
(1) a personal story describing sexual violence and 
(2) {n} responses offering support or guidance. 
Your job is to evaluate the responses as replies to the storyteller.

Task: Rank the responses by overall usefulness and reliability based only on the text provided. Do not assume facts that are not stated.

TIES ARE ALLOWED: Example (5 comments): Ranking: 2,1,3,3,5
This means comment [1] is ranked 2nd, comment [2] is ranked 1st, comments [3] and [4] are tied for 3rd, and comment [5] is ranked 5th.

IMPORTANT OUTPUT FORMAT:
Return EXACTLY one line in this format:
Ranking: <your ranking>
No extra text.

STORY:
{story}

RESPONSES:
{responses}
"""

def build_prompt(story: str, comments: list[str]) -> str:
    clean = []
    for c in comments:
        if c is None:
            continue
        s = str(c).strip()
        if s and s.lower() != "nan":
            clean.append(s)

    responses_block = "\n".join([f"[{i}] {c}" for i, c in enumerate(clean, start=1)])

    return PROMPT_TEMPLATE.format(
        n=len(clean),
        story=str(story).strip(),
        responses=responses_block
    )

In [7]:
MODEL = "gpt-5.2"
TEMPERATURE = 0
MAX_OUTPUT_TOKENS = 80

def call_ranker(prompt: str) -> str:
    last_err = None
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "You are a careful evaluator of supportive responses. Follow the output format exactly."},
                    {"role": "user", "content": prompt},
                ],
                temperature=TEMPERATURE,
                max_completion_tokens=MAX_OUTPUT_TOKENS,
                timeout=60,
            )
            text = resp.choices[0].message.content.strip()
            if not text.startswith("Ranking:"):
                raise ValueError(f"Bad format: {text[:120]}")
            return text
        except Exception as e:
            last_err = e
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"LLM call failed after retries: {last_err}")


In [8]:
RANKING_RE = re.compile(r"^Ranking:\s*(.+)\s*$")

def parse_ranking_line(line: str, n: int) -> dict:
    """
    Expects: Ranking: r1,r2,...,rn
    where ri is the rank (1..n) assigned to comment i. Ties allowed via duplicate ranks.

    Returns:
      {
        "raw": str,
        "ranking": str|None,
        "valid": bool,
        "error": str|None,
        "ranks": list[int]|None,          # length n, ranks per comment index
        "order_groups": list[list[int]]|None  # comment indices grouped by rank (best rank=1 first)
      }
    """
    out = {"raw": line, "ranking": None, "valid": False, "error": None,
           "ranks": None, "order_groups": None}

    m = RANKING_RE.match(line.strip())
    if not m:
        out["error"] = "Missing/invalid 'Ranking:' prefix"
        return out

    ranking = m.group(1).strip()
    out["ranking"] = ranking

    parts = [p.strip() for p in ranking.split(",") if p.strip()]
    if len(parts) != n:
        out["error"] = f"Expected exactly {n} comma-separated integers, got {len(parts)}"
        return out

    ranks = []
    for p in parts:
        if not p.isdigit():
            out["error"] = f"Non-integer token: '{p}'"
            return out
        r = int(p)
        if r < 1 or r > n:
            out["error"] = f"Rank out of range 1..{n}: {r}"
            return out
        ranks.append(r)

    out["ranks"] = ranks

    # Convert to ordered groups of comment indices by rank (1 is best)
    rank_to_indices = {}
    for idx, r in enumerate(ranks, start=1):  # comment indices are 1-based
        rank_to_indices.setdefault(r, []).append(idx)

    # Ensure ranking uses rank 1 at least once (optional but usually desired)
    if 1 not in rank_to_indices:
        out["error"] = "Ranking must include rank 1 at least once"
        out["order_groups"] = [rank_to_indices[r] for r in sorted(rank_to_indices)]
        return out

    out["order_groups"] = [rank_to_indices[r] for r in sorted(rank_to_indices)]
    out["valid"] = True
    return out


In [9]:
# Pick any row index to test
test_i = 0

row = df.iloc[test_i]
story = get_story_text(row)
comments = get_comments_list(row)

print("post_id:", row.get("post_id"))
print("Num comments found:", len(comments))

bp = build_prompt(story, comments)
if isinstance(bp, tuple):
    prompt, comments_clean = bp
    # Use the cleaned list for n
    n = len(comments_clean)
else:
    prompt = bp
    n = len([c for c in comments if c is not None and str(c).strip() and str(c).strip().lower() != "nan"])

print(prompt[:800], "\n...\n")

res = call_ranker(prompt)
print("MODEL OUTPUT:", res)

parsed = parse_ranking_line(res, n=n)
parsed


post_id: 1hv4zdy
Num comments found: 3


I will give you 
(1) a personal story describing sexual violence and 
(2) 3 responses offering support or guidance. 
Your job is to evaluate the responses as replies to the storyteller.

Task: Rank the responses by overall usefulness and reliability based only on the text provided. Do not assume facts that are not stated.

TIES ARE ALLOWED: Example (5 comments): Ranking: 2,1,3,3,5
This means comment [1] is ranked 2nd, comment [2] is ranked 1st, comments [3] and [4] are tied for 3rd, and comment [5] is ranked 5th.

IMPORTANT OUTPUT FORMAT:
Return EXACTLY one line in this format:
Ranking: <your ranking>
No extra text.

STORY:
1hv4zdy meToo Was this SA? Ok-Sugar959 /r/meToo/comments/1hv4zdy/was_this_sa/ So l've been in a relationship with a girl who has bpd. We had alot of ups and downs but 
...

MODEL OUTPUT: Ranking: 3,2,1


{'raw': 'Ranking: 3,2,1',
 'ranking': '3,2,1',
 'valid': True,
 'error': None,
 'ranks': [3, 2, 1],
 'order_groups': [[3], [2], [1]]}

In [10]:
# Initialize as string/object dtype to avoid FutureWarning
df["Human_Ranking"] = pd.Series([None] * len(df), dtype="object")
df["LLM_Ranking"] = pd.Series([None] * len(df), dtype="object")
df["LLM_Ranking_Error"] = pd.Series([None] * len(df), dtype="object")

# bool column
df["LLM_Ranking_Valid"] = False


In [11]:
START_ROW = 0
END_ROW = len(df)          # whole file
SLEEP_BETWEEN = 0.3        # pacing

# Prepare output columns
df["Human_Ranking"] = np.nan
df["LLM_Ranking"] = np.nan
df["LLM_Ranking_Valid"] = False
df["LLM_Ranking_Error"] = np.nan

for idx in tqdm(range(START_ROW, END_ROW)):
    row = df.iloc[idx]
    story = get_story_text(row)
    comments = get_comments_list(row)
    n = len(comments)

    # Human ranking always just 1..n (even if n is 0/1)
    if n > 0:
        df.at[idx, "Human_Ranking"] = ",".join(str(i) for i in range(1, n + 1))
    else:
        df.at[idx, "Human_Ranking"] = ""  # or leave NaN if you prefer

    # If no story, skip LLM
    if not story or str(story).strip() == "":
        df.at[idx, "LLM_Ranking_Error"] = "Skipped (missing story text)"
        continue

    # If fewer than 2 comments, nothing to rank
    if n < 2:
        df.at[idx, "LLM_Ranking_Error"] = f"Skipped (n_comments_used={n})"
        continue

    prompt = build_prompt(story, comments)

    try:
        raw = call_ranker(prompt)
        parsed = parse_ranking_line(raw, n=n)

        df.at[idx, "LLM_Ranking"] = parsed["ranking"] if parsed["ranking"] is not None else raw
        df.at[idx, "LLM_Ranking_Valid"] = bool(parsed["valid"])
        df.at[idx, "LLM_Ranking_Error"] = parsed["error"]

        # If invalid, keep the raw output for debugging (optional)
        if not parsed["valid"] and pd.isna(df.at[idx, "LLM_Ranking"]):
            df.at[idx, "LLM_Ranking"] = raw

    except Exception as e:
        df.at[idx, "LLM_Ranking_Error"] = str(e)

    time.sleep(SLEEP_BETWEEN)

OUT_PATH = "Ambivalent_Binning_with_Rankings.csv"
df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

df[["post_id", "Human_Ranking", "LLM_Ranking", "LLM_Ranking_Valid", "LLM_Ranking_Error"]].head(10)


  0%|          | 0/1082 [00:00<?, ?it/s]/tmp/ipykernel_27168/360010796.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1,2,3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "Human_Ranking"] = ",".join(str(i) for i in range(1, n + 1))
/tmp/ipykernel_27168/360010796.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3,2,1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "LLM_Ranking"] = parsed["ranking"] if parsed["ranking"] is not None else raw
  0%|          | 2/1082 [00:01<16:59,  1.06it/s]/tmp/ipykernel_27168/360010796.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Skipped (n_comments_used=1)' has dtype incompatible wit

Saved: Ambivalent_Binning_with_Rankings.csv


,post_id,Human_Ranking,LLM_Ranking,LLM_Ranking_Valid,LLM_Ranking_Error
0,1hv4zdy,"1,2,3","3,2,1",True,NaN
1,1f2kibx,"1,2","2,1",True,NaN
2,1ccnt06,1,NaN,False,Skipped (n_comments_used=1)
3,1c09z00,1,NaN,False,Skipped (n_comments_used=1)
4,1bautv8,"1,2","2,1",True,None
5,1av6815,1,NaN,False,Skipped (n_comments_used=1)
6,15otbl9,"1,2","2,1",True,None
7,14cey9m,1,NaN,False,Skipped (n_comments_used=1)
8,11lre68,1,NaN,False,Skipped (n_comments_used=1)
9,11f4o14,"1,2,3","1,2,3",True,None


In [12]:
print("df columns sample:", df.columns.tolist()[:40])


df columns sample: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'bin01_comment', 'bin02_comment', 'bin03_comment', 'bin04_comment', 'bin05_comment', 'bin06_comment', 'bin07_comment', 'bin08_comment', 'bin09_comment', 'bin10_comment', 'bin11_comment', 'bin12_comment', 'bin13_comment', 'bin14_comment', 'bin15_comment', 'bin16_comment', 'bin17_comment', 'bin18_comment', 'bin19_comment', 'bin20_comment', 'bin21_comment', 'bin22_comment', 'Human_Ranking', 'LLM_Ranking', 'LLM_Ranking_Error', 'LLM_Ranking_Valid']


In [27]:
import pandas as pd
import numpy as np

IN_PATH = "Ambivalent_Binning_with_Rankings_2.csv"
OUT_PATH = "Ambivalent_Binning_with_Rankings_3.csv"

df = pd.read_csv(IN_PATH)

def reverse_ranking(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return s
    txt = str(s).strip()
    if txt == "" or txt.lower() == "nan":
        return txt
    parts = [p.strip() for p in txt.split(",") if p.strip() != ""]
    parts_rev = list(reversed(parts))
    return ",".join(parts_rev)

df["Human_Ranking"] = df["Human_Ranking"].apply(reverse_ranking)

df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

# quick check
print(df[["post_id", "Human_Ranking"]].head(5).to_string(index=False))


Saved: Ambivalent_Binning_with_Rankings_3.csv
post_id Human_Ranking
1hv4zdy         3,2,1
1f2kibx           2,1
1bautv8           2,1
15otbl9           2,1
11f4o14         3,2,1


In [28]:
%pip -q install pingouin

You should consider upgrading via the '/home/ec2-user/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [29]:
import pandas as pd

CSV_PATH = "Ambivalent_Binning_with_Rankings_3.csv"
df = pd.read_csv(CSV_PATH)

# If you saved the ranked output already, use THAT instead:
# df = pd.read_csv("/mnt/data/Ambivalent_posts_with_top10_comments_with_rankings.csv")

df_out = df.copy()

In [30]:
df_out.columns = (
    df_out.columns.astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

required = ["post_id", "Human_Ranking", "LLM_Ranking"]
missing = [c for c in required if c not in df_out.columns]
print("Missing columns:", missing)

df_out[required].head(5)


Missing columns: []


,post_id,Human_Ranking,LLM_Ranking
0,1hv4zdy,"3,2,1","3,2,1"
1,1f2kibx,"2,1","2,1"
2,1bautv8,"2,1","2,1"
3,15otbl9,"2,1","2,1"
4,11f4o14,"3,2,1","1,2,3"


In [31]:
import numpy as np

def parse_rank_vector(raw: str, n: int):
    """
    Parse rank-vector encoding of length n, e.g.:
      "1,1,3" or "1.0, 1.0, 3.0" or "[1, 1, 3]" or "Ranking: 1,1,3"
    Returns dict {item_index: rank_value} for item_index in 1..n.
    """
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        raise ValueError("Missing ranking")

    s = str(raw).strip()
    if s == "" or s.lower() == "nan":
        raise ValueError("Empty ranking")

    s = s.replace("Ranking:", "").strip()
    s = s.strip("[](){}")

    parts = [p.strip() for p in s.split(",") if p.strip() != ""]
    if len(parts) != n:
        raise ValueError(f"Expected {n} numbers but got {len(parts)} from '{raw}'")

    vals = [float(p) for p in parts]
    return {i: float(vals[i-1]) for i in range(1, n + 1)}

def rank_norm(rank_val: float, n: int) -> float:
    """
    Normalize ranks so different n are comparable across posts:
      best -> 0, worst -> 1
    """
    if n <= 1:
        return np.nan
    return (float(rank_val) - 1.0) / (n - 1.0)


In [32]:
use = df_out.dropna(subset=["post_id", "Human_Ranking", "LLM_Ranking"]).copy()

rows = []
bad_rows = 0

for _, r in use.iterrows():
    pid = str(r["post_id"])

    # n = number of ranks in Human_Ranking (since it's always "1,2,...,n")
    try:
        n = len([p for p in str(r["Human_Ranking"]).split(",") if p.strip() != ""])
        if n < 2:
            continue
    except Exception:
        bad_rows += 1
        continue

    try:
        hr = parse_rank_vector(r["Human_Ranking"], n)
        lr = parse_rank_vector(r["LLM_Ranking"], n)
    except Exception:
        bad_rows += 1
        continue

    for item in range(1, n + 1):
        target = f"{pid}_resp{item}"
        rows.append({"target": target, "rater": "human", "rating": rank_norm(hr[item], n)})
        rows.append({"target": target, "rater": "llm",   "rating": rank_norm(lr[item], n)})

long_df = pd.DataFrame(rows, columns=["target", "rater", "rating"])

print("Rows considered:", len(use))
print("Skipped rows (parse issues or n<2):", bad_rows)
print("long_df shape:", long_df.shape)
long_df.head(10)


Rows considered: 605
Skipped rows (parse issues or n<2): 0
long_df shape: (3254, 3)


,target,rater,rating
0,1hv4zdy_resp1,human,1.0
1,1hv4zdy_resp1,llm,1.0
2,1hv4zdy_resp2,human,0.5
3,1hv4zdy_resp2,llm,0.5
4,1hv4zdy_resp3,human,0.0
5,1hv4zdy_resp3,llm,0.0
6,1f2kibx_resp1,human,1.0
7,1f2kibx_resp1,llm,1.0
8,1f2kibx_resp2,human,0.0
9,1f2kibx_resp2,llm,0.0


In [33]:
import pingouin as pg

icc_tbl = pg.intraclass_corr(
    data=long_df,
    targets="target",
    raters="rater",
    ratings="rating"
)

icc_tbl


,Type,Description,ICC,F,df1,df2,pval,CI95%
0,ICC1,Single raters absolute,0.316900,1.927831,1626,1627,1.288276e-39,"[0.27, 0.36]"
1,ICC2,Single random raters,0.316767,1.926728,1626,1626,1.543005e-39,"[0.27, 0.36]"
2,ICC3,Single fixed raters,0.316643,1.926728,1626,1626,1.543005e-39,"[0.27, 0.36]"
3,ICC1k,Average raters absolute,0.481282,1.927831,1626,1627,1.288276e-39,"[0.43, 0.53]"
4,ICC2k,Average random raters,0.481128,1.926728,1626,1626,1.543005e-39,"[0.43, 0.53]"
5,ICC3k,Average fixed raters,0.480986,1.926728,1626,1626,1.543005e-39,"[0.43, 0.53]"


In [34]:
icc_tbl[icc_tbl["Type"].isin(["ICC2", "ICC3"])]


,Type,Description,ICC,F,df1,df2,pval,CI95%
1,ICC2,Single random raters,0.316767,1.926728,1626,1626,1.543005e-39,"[0.27, 0.36]"
2,ICC3,Single fixed raters,0.316643,1.926728,1626,1626,1.543005e-39,"[0.27, 0.36]"


In [35]:
icc2 = icc_tbl.loc[icc_tbl["Type"] == "ICC2"].iloc[0]
icc3 = icc_tbl.loc[icc_tbl["Type"] == "ICC3"].iloc[0]

print("Pooled ICC on normalized ranks (human vs LLM):\n")

print("ICC(2,1) [ICC2] — absolute agreement (two-way random effects):")
print("  ICC :", float(icc2["ICC"]))
print("  p   :", float(icc2["pval"]))
print("  CI95:", icc2["CI95%"])
print()

print("ICC(3,1) [ICC3] — consistency (two-way mixed effects):")
print("  ICC :", float(icc3["ICC"]))
print("  p   :", float(icc3["pval"]))
print("  CI95:", icc3["CI95%"])


Pooled ICC on normalized ranks (human vs LLM):

ICC(2,1) [ICC2] — absolute agreement (two-way random effects):
  ICC : 0.31676688687079674
  p   : 1.543005411142962e-39
  CI95: [0.27 0.36]

ICC(3,1) [ICC3] — consistency (two-way mixed effects):
  ICC : 0.31664313346290635
  p   : 1.543005411142962e-39
  CI95: [0.27 0.36]


In [36]:
pip install scipy

You should consider upgrading via the '/home/ec2-user/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [38]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

IN_PATH = "Ambivalent_Binning_with_Rankings_3.csv"
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)

H_COL = "Human_Ranking"
L_COL = "LLM_Ranking"

def extract_ints(s: str):
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    if pd.isna(s):
        raise ValueError("Ranking is NaN/empty")
    nums = extract_ints(s)
    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")
    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")
    return nums

def infer_n_from_row(row):
    # Prefer n_comments_used if present
    n = row.get("n_comments_used", np.nan)
    if not pd.isna(n):
        try:
            n = int(float(n))
            if n > 0:
                return n
        except Exception:
            pass

    # fallback from human ranking length
    h = row.get(H_COL, None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback from llm ranking length
    m = row.get(L_COL, None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n.")

def compute_ira_row(row):
    n = infer_n_from_row(row)

    human = parse_rankvec(row[H_COL], n)
    llm   = parse_rankvec(row[L_COL], n)

    tau, tau_p = kendalltau(human, llm, variant="b")  # tie-aware
    rho, rho_p = spearmanr(human, llm)                # tie-aware

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# 3) Compute IRA per row
required = {H_COL, L_COL}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# 4) Summary (overall IRA)
valid = df_with_ira["ira_valid"] == True

# Optional: if you have LLM_Ranking_Valid column, AND it exists, use it too
if "LLM_Ranking_Valid" in df_with_ira.columns:
    valid = valid & (df_with_ira["LLM_Ranking_Valid"] == True)

print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b:", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

if (~valid).any():
    cols_to_show = [c for c in ["post_id", "n_comments_used", H_COL, L_COL, "ira_error"] if c in df_with_ira.columns]
    print("\nSample invalid rows:")
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# 5) Save
OUT_PATH = "Ambivalent_Binning_with_IRA.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: Ambivalent_Binning_with_Rankings_3.csv
Shape: (605, 38)


/tmp/ipykernel_27168/4125025700.py:62: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, rho_p = spearmanr(human, llm)                # tie-aware



Valid rows: 605 / 605
Mean Kendall Tau-b: 0.3057706873306463
Mean Spearman Rho: 0.31884625770621255

Saved: Ambivalent_Binning_with_IRA.csv


## IRA

In [ ]:
pip install scipy

     |████████████████████████████████| 38.6 MB 59.2 MB/s            ██████▎                | 18.4 MB 8.2 MB/s eta 0:00:03
You should consider upgrading via the '/home/ec2-user/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [48]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

# =========================
# 1) Load input
# =========================
IN_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50.csv"

# If your file is tab-separated, switch to: pd.read_csv(IN_PATH, sep="\t")
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# =========================
# 2) Helpers
# =========================
def extract_ints(s: str):
    """Extract integers from a string like '1,1,3' or 'Ranking: 1, 1, 3' or '[1,1,3]'."""
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    """
    Parse a per-comment rank vector from a cell like:
      "1,1,3" or "1, 1, 3" or "[1,1,3]" or "Ranking: 1, 1, 3"
    Returns: list[int] of length n
    """
    if pd.isna(s):
        raise ValueError("Ranking is NaN")

    nums = extract_ints(s)

    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")

    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")

    return nums

def infer_n_from_rankings(row):
    """
    Prefer n_comments_used if present/valid, otherwise infer from Human_Rankings length,
    otherwise infer from LLM_Rankings length.
    """
    n = row.get("n_comments_used", np.nan)

    if not pd.isna(n):
        try:
            n = int(n)
            if n > 0:
                return n
        except Exception:
            pass

    # fallback: infer from human ranking length
    h = row.get("Human_Rankings", None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback: infer from llm ranking length
    m = row.get("LLM_Rankings", None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n (n_comments_used missing/invalid and rankings empty).")

def compute_ira_row(row):
    n = infer_n_from_rankings(row)

    human = parse_rankvec(row["Human_Rankings"], n)
    llm   = parse_rankvec(row["LLM_Rankings"], n)

    # Kendall tau-b handles ties properly
    tau, tau_p = kendalltau(human, llm, variant="b")

    # Spearman rho (also supports ties)
    rho, rho_p = spearmanr(human, llm)

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# =========================
# 3) Compute IRA per row
# =========================
required = {"Human_Rankings", "LLM_Rankings"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# =========================
# 4) Summary (overall IRA)
# =========================
valid = df_with_ira["ira_valid"] == True
print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b (IRA):", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

# Show a few failures (if any)
if (~valid).any():
    print("\nSample invalid rows:")
    cols_to_show = [c for c in ["post_id", "n_comments_used", "Human_Rankings", "LLM_Rankings", "ira_error"] if c in df_with_ira.columns]
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# =========================
# 5) Save
# =========================
OUT_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50_with_IRA.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50.csv
Shape: (195, 31)
Columns: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'matched_ambivalent_phrases', 'matched_keywords', 'TC1', 'TC2', 'TC3', 'TC4', 'TC5', 'TC6', 'TC7', 'TC8', 'TC9', 'TC10', 'row_index', 'n_comments_used', 'ranking_raw', 'ranking_valid', 'ranking_error', 'Human_Rankings', 'LLM_Rankings']

Valid rows: 195 / 195
Mean Kendall Tau-b (IRA): 0.1967844147521091
Mean Spearman Rho: 0.19833756375580863

Saved: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50_with_IRA.csv


In [49]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

# =========================
# 1) Load input
# =========================
IN_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_50_75.csv"

# If your file is tab-separated, switch to: pd.read_csv(IN_PATH, sep="\t")
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# =========================
# 2) Helpers
# =========================
def extract_ints(s: str):
    """Extract integers from a string like '1,1,3' or 'Ranking: 1, 1, 3' or '[1,1,3]'."""
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    """
    Parse a per-comment rank vector from a cell like:
      "1,1,3" or "1, 1, 3" or "[1,1,3]" or "Ranking: 1, 1, 3"
    Returns: list[int] of length n
    """
    if pd.isna(s):
        raise ValueError("Ranking is NaN")

    nums = extract_ints(s)

    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")

    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")

    return nums

def infer_n_from_rankings(row):
    """
    Prefer n_comments_used if present/valid, otherwise infer from Human_Rankings length,
    otherwise infer from LLM_Rankings length.
    """
    n = row.get("n_comments_used", np.nan)

    if not pd.isna(n):
        try:
            n = int(n)
            if n > 0:
                return n
        except Exception:
            pass

    # fallback: infer from human ranking length
    h = row.get("Human_Rankings", None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback: infer from llm ranking length
    m = row.get("LLM_Rankings", None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n (n_comments_used missing/invalid and rankings empty).")

def compute_ira_row(row):
    n = infer_n_from_rankings(row)

    human = parse_rankvec(row["Human_Rankings"], n)
    llm   = parse_rankvec(row["LLM_Rankings"], n)

    # Kendall tau-b handles ties properly
    tau, tau_p = kendalltau(human, llm, variant="b")

    # Spearman rho (also supports ties)
    rho, rho_p = spearmanr(human, llm)

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# =========================
# 3) Compute IRA per row
# =========================
required = {"Human_Rankings", "LLM_Rankings"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# =========================
# 4) Summary (overall IRA)
# =========================
valid = df_with_ira["ira_valid"] == True
print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b (IRA):", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

# Show a few failures (if any)
if (~valid).any():
    print("\nSample invalid rows:")
    cols_to_show = [c for c in ["post_id", "n_comments_used", "Human_Rankings", "LLM_Rankings", "ira_error"] if c in df_with_ira.columns]
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# =========================
# 5) Save
# =========================
OUT_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_50_75_with_IRA.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_50_75.csv
Shape: (194, 31)
Columns: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'matched_ambivalent_phrases', 'matched_keywords', 'TC1', 'TC2', 'TC3', 'TC4', 'TC5', 'TC6', 'TC7', 'TC8', 'TC9', 'TC10', 'row_index', 'n_comments_used', 'ranking_raw', 'ranking_valid', 'ranking_error', 'Human_Rankings', 'LLM_Rankings']

Valid rows: 194 / 194
Mean Kendall Tau-b (IRA): 0.2660290288746958
Mean Spearman Rho: 0.290542857411822

Saved: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_50_75_with_IRA.csv


In [50]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

# =========================
# 1) Load input
# =========================
IN_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_75_100.csv"

# If your file is tab-separated, switch to: pd.read_csv(IN_PATH, sep="\t")
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# =========================
# 2) Helpers
# =========================
def extract_ints(s: str):
    """Extract integers from a string like '1,1,3' or 'Ranking: 1, 1, 3' or '[1,1,3]'."""
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    """
    Parse a per-comment rank vector from a cell like:
      "1,1,3" or "1, 1, 3" or "[1,1,3]" or "Ranking: 1, 1, 3"
    Returns: list[int] of length n
    """
    if pd.isna(s):
        raise ValueError("Ranking is NaN")

    nums = extract_ints(s)

    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")

    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")

    return nums

def infer_n_from_rankings(row):
    """
    Prefer n_comments_used if present/valid, otherwise infer from Human_Rankings length,
    otherwise infer from LLM_Rankings length.
    """
    n = row.get("n_comments_used", np.nan)

    if not pd.isna(n):
        try:
            n = int(n)
            if n > 0:
                return n
        except Exception:
            pass

    # fallback: infer from human ranking length
    h = row.get("Human_Rankings", None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback: infer from llm ranking length
    m = row.get("LLM_Rankings", None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n (n_comments_used missing/invalid and rankings empty).")

def compute_ira_row(row):
    n = infer_n_from_rankings(row)

    human = parse_rankvec(row["Human_Rankings"], n)
    llm   = parse_rankvec(row["LLM_Rankings"], n)

    # Kendall tau-b handles ties properly
    tau, tau_p = kendalltau(human, llm, variant="b")

    # Spearman rho (also supports ties)
    rho, rho_p = spearmanr(human, llm)

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# =========================
# 3) Compute IRA per row
# =========================
required = {"Human_Rankings", "LLM_Rankings"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# =========================
# 4) Summary (overall IRA)
# =========================
valid = df_with_ira["ira_valid"] == True
print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b (IRA):", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

# Show a few failures (if any)
if (~valid).any():
    print("\nSample invalid rows:")
    cols_to_show = [c for c in ["post_id", "n_comments_used", "Human_Rankings", "LLM_Rankings", "ira_error"] if c in df_with_ira.columns]
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# =========================
# 5) Save
# =========================
OUT_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_75_100_with_IRA.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_75_100.csv
Shape: (160, 31)
Columns: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'matched_ambivalent_phrases', 'matched_keywords', 'TC1', 'TC2', 'TC3', 'TC4', 'TC5', 'TC6', 'TC7', 'TC8', 'TC9', 'TC10', 'row_index', 'n_comments_used', 'ranking_raw', 'ranking_valid', 'ranking_error', 'Human_Rankings', 'LLM_Rankings']

Valid rows: 159 / 160
Mean Kendall Tau-b (IRA): 0.37701113492725724
Mean Spearman Rho: 0.4281988516911145

Sample invalid rows:
post_id  n_comments_used Human_Rankings LLM_Rankings                               ira_error
11t1ste                3          1,2,2      2,3,3,1 Expected 3 ranks, got 4 -> [2, 3, 3, 1]

Saved: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_75_100_with_IRA.csv


## Infra class correlation

In [17]:
%pip -q install pingouin

You should consider upgrading via the '/home/ec2-user/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [43]:
import pandas as pd

CSV_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50_with_IRA.csv"
df = pd.read_csv(CSV_PATH)

# If you saved the ranked output already, use THAT instead:
# df = pd.read_csv("/mnt/data/Ambivalent_posts_with_top10_comments_with_rankings.csv")

df_out = df.copy()

In [44]:
df_out.columns = (
    df_out.columns.astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

required = ["post_id", "Human_Rankings", "LLM_Rankings", "n_used_for_ira"]
missing = [c for c in required if c not in df_out.columns]
print("Missing columns:", missing)

df_out[required].head(5)


Missing columns: []


,post_id,Human_Rankings,LLM_Rankings,n_used_for_ira
0,1hv4zdy,"1,1,3","3,2,1",3
1,11f4o14,"1,2","2,1",2
2,xyf4l9,"1,1,3","2,1,3",3
3,pihi7p,"1,1,3,3","1,4,3,2",4
4,1ofr7sg,"1,2","2,1",2


In [45]:
def parse_rank_vector(raw: str, n: int):
    """
    Parse rank-vector encoding of length n, e.g.:
      "1,1,3" or "1.0, 1.0, 3.0" or "[1, 1, 3]"
    Returns dict {item_index: rank_value} for item_index in 1..n.
    """
    s = str(raw).strip()
    s = s.replace("Ranking:", "").strip()
    s = s.strip("[](){}")

    parts = [p.strip() for p in s.split(",") if p.strip() != ""]
    if len(parts) != n:
        raise ValueError(f"Expected {n} numbers but got {len(parts)} from '{raw}'")

    vals = [float(p) for p in parts]
    return {i: float(vals[i-1]) for i in range(1, n + 1)}

def rank_norm(rank_val: float, n: int) -> float:
    """
    Normalize ranks so different n are comparable across posts:
      best -> 0, worst -> 1
    """
    if n <= 1:
        return np.nan
    return (float(rank_val) - 1.0) / (n - 1.0)


In [46]:
use = df_out.dropna(subset=["post_id", "Human_Rankings", "LLM_Rankings", "n_used_for_ira"]).copy()

rows = []
bad_rows = 0

for _, r in use.iterrows():
    pid = str(r["post_id"])
    n = int(float(r["n_used_for_ira"]))  # robust if stored as 3.0

    try:
        hr = parse_rank_vector(r["Human_Rankings"], n)
        lr = parse_rank_vector(r["LLM_Rankings"], n)
    except Exception:
        bad_rows += 1
        continue

    for item in range(1, n + 1):
        target = f"{pid}_resp{item}"
        rows.append({"target": target, "rater": "human", "rating": rank_norm(hr[item], n)})
        rows.append({"target": target, "rater": "llm",   "rating": rank_norm(lr[item], n)})

long_df = pd.DataFrame(rows, columns=["target", "rater", "rating"])

print("Rows used:", len(use))
print("Skipped rows (parse issues):", bad_rows)
print("long_df shape:", long_df.shape)
long_df.head(10)


Rows used: 195
Skipped rows (parse issues): 0
long_df shape: (1006, 3)


,target,rater,rating
0,1hv4zdy_resp1,human,0.0
1,1hv4zdy_resp1,llm,1.0
2,1hv4zdy_resp2,human,0.0
3,1hv4zdy_resp2,llm,0.5
4,1hv4zdy_resp3,human,1.0
5,1hv4zdy_resp3,llm,0.0
6,11f4o14_resp1,human,0.0
7,11f4o14_resp1,llm,1.0
8,11f4o14_resp2,human,1.0
9,11f4o14_resp2,llm,0.0


In [47]:
icc_tbl = pg.intraclass_corr(
    data=long_df,
    targets="target",
    raters="rater",
    ratings="rating"
)

icc_tbl


,Type,Description,ICC,F,df1,df2,pval,CI95%
0,ICC1,Single raters absolute,0.135662,1.313909,388,389,0.003628,"[0.04, 0.23]"
1,ICC2,Single random raters,0.145682,1.350563,388,388,0.001577,"[0.05, 0.24]"
2,ICC3,Single fixed raters,0.149140,1.350563,388,388,0.001577,"[0.05, 0.24]"
3,ICC1k,Average raters absolute,0.238912,1.313909,388,389,0.003628,"[0.07, 0.38]"
4,ICC2k,Average random raters,0.254315,1.350563,388,388,0.001577,"[0.09, 0.39]"
5,ICC3k,Average fixed raters,0.259568,1.350563,388,388,0.001577,"[0.1, 0.39]"


In [48]:
icc_tbl[icc_tbl["Type"].isin(["ICC2", "ICC3"])]


,Type,Description,ICC,F,df1,df2,pval,CI95%
1,ICC2,Single random raters,0.145682,1.350563,388,388,0.001577,"[0.05, 0.24]"
2,ICC3,Single fixed raters,0.149140,1.350563,388,388,0.001577,"[0.05, 0.24]"


In [49]:
icc2 = icc_tbl.loc[icc_tbl["Type"] == "ICC2"].iloc[0]
icc3 = icc_tbl.loc[icc_tbl["Type"] == "ICC3"].iloc[0]

print("ICC ranks for 25-50%:")

print("Pooled ICC(2,1) [ICC2] absolute agreement on normalized ranks:")
print("  ICC :", float(icc2["ICC"]))
print("  p   :", float(icc2["pval"]))
print("  CI95:", icc2["CI95%"])
print()

print("Pooled ICC(3,1) [ICC3] absolute agreement on normalized ranks:")
print("  ICC :", float(icc3["ICC"]))
print("  p   :", float(icc3["pval"]))
print("  CI95:", icc3["CI95%"])


ICC ranks for 25-50%:
Pooled ICC(2,1) [ICC2] absolute agreement on normalized ranks:
  ICC : 0.14568206052529056
  p   : 0.0015768033972234402
  CI95: [0.05 0.24]

Pooled ICC(3,1) [ICC3] absolute agreement on normalized ranks:
  ICC : 0.14914004819352575
  p   : 0.0015768033972234402
  CI95: [0.05 0.24]


In [34]:
icc2 = icc_tbl.loc[icc_tbl["Type"] == "ICC2"].iloc[0]
icc3 = icc_tbl.loc[icc_tbl["Type"] == "ICC3"].iloc[0]

print("ICC ranks for 50-75%:")

print("Pooled ICC(2,1) [ICC2] absolute agreement on normalized ranks:")
print("  ICC :", float(icc2["ICC"]))
print("  p   :", float(icc2["pval"]))
print("  CI95:", icc2["CI95%"])
print()

print("Pooled ICC(3,1) [ICC3] absolute agreement on normalized ranks:")
print("  ICC :", float(icc3["ICC"]))
print("  p   :", float(icc3["pval"]))
print("  CI95:", icc3["CI95%"])


ICC ranks for 50-75%:
Pooled ICC(2,1) [ICC2] absolute agreement on normalized ranks:
  ICC : 0.30941422850865863
  p   : 3.124683562865803e-14
  CI95: [0.23 0.38]

Pooled ICC(3,1) [ICC3] absolute agreement on normalized ranks:
  ICC : 0.31421756930548084
  p   : 3.124683562865803e-14
  CI95: [0.24 0.39]


In [42]:
icc2 = icc_tbl.loc[icc_tbl["Type"] == "ICC2"].iloc[0]
icc3 = icc_tbl.loc[icc_tbl["Type"] == "ICC3"].iloc[0]

print("ICC ranks for 75-100%:")

print("Pooled ICC(2,1) [ICC2] absolute agreement on normalized ranks:")
print("  ICC :", float(icc2["ICC"]))
print("  p   :", float(icc2["pval"]))
print("  CI95:", icc2["CI95%"])
print()

print("Pooled ICC(3,1) [ICC3] absolute agreement on normalized ranks:")
print("  ICC :", float(icc3["ICC"]))
print("  p   :", float(icc3["pval"]))
print("  CI95:", icc3["CI95%"])


ICC ranks for 75-100%:
Pooled ICC(2,1) [ICC2] absolute agreement on normalized ranks:
  ICC : 0.40158314445410276
  p   : 3.4394248315387075e-30
  CI95: [0.34 0.46]

Pooled ICC(3,1) [ICC3] absolute agreement on normalized ranks:
  ICC : 0.40330914499152115
  p   : 3.4394248315387075e-30
  CI95: [0.34 0.46]


In [16]:
import numpy as np
import pandas as pd

def parse_rank_vector_if_possible(raw: str, n: int):
    """
    Try to interpret raw as a rank-vector of length n.
    Supports:
      "1,1,3"
      "1.0, 1.0, 3.0"
      "[1, 1, 3]"
      "(1,1,3)"
    """
    s = str(raw).strip()

    # remove common wrappers
    s = s.replace("Ranking:", "").strip()
    s = s.strip("[](){}")

    # split on commas
    parts = [p.strip() for p in s.split(",") if p.strip() != ""]
    if len(parts) != n:
        return None

    # allow floats like "1.0"
    vals = []
    for p in parts:
        try:
            vals.append(float(p))
        except:
            return None

    # return as ranks per item index 1..n
    return {i: float(vals[i-1]) for i in range(1, n+1)}

def parse_ordered_ids(raw: str, n: int):
    """
    Ordered IDs format like:
      "3=1, 2"
      "3,2,1"
    Returns ranks per item index 1..n.
    """
    s = str(raw).strip()
    s = s.replace("Ranking:", "").strip()

    chunks = [c.strip() for c in s.split(",") if c.strip()]
    if not chunks:
        raise ValueError("empty ranking")

    groups = []
    for c in chunks:
        if "=" in c:
            tied = [int(x) for x in re.findall(r"\d+", c)]
            if tied:
                groups.append(tied)
        else:
            nums = [int(x) for x in re.findall(r"\d+", c)]
            # sequential items
            for num in nums:
                groups.append([num])

    ranks = {}
    current_rank = 1
    for tied in groups:
        k = len(tied)
        avg_rank = (current_rank + (current_rank + k - 1)) / 2.0
        for item in tied:
            ranks[item] = avg_rank
        current_rank += k

    if set(ranks.keys()) != set(range(1, n + 1)):
        raise ValueError(f"ordered ids does not cover 1..{n}. keys={sorted(ranks.keys())}")

    return ranks

def parse_ranks_any_v2(raw: str, n: int):
    # 1) Try rank-vector
    rv = parse_rank_vector_if_possible(raw, n)
    if rv is not None:
        return rv
    # 2) Fall back to ordered IDs
    return parse_ordered_ids(raw, n)

def rank_norm(rank_val: float, n: int) -> float:
    # best -> 0, worst -> 1
    if n <= 1:
        return np.nan
    return (float(rank_val) - 1.0) / (n - 1.0)

rows = []
bad_rows = 0
bad_examples = []

for _, r in use.iterrows():
    pid = str(r["post_id"])
    n = int(float(r["n_used_for_ira"]))  # handles 3.0 safely

    try:
        hr = parse_ranks_any_v2(r["Human_Rankings"], n)
        lr = parse_ranks_any_v2(r["LLM_Rankings"], n)
    except Exception as e:
        bad_rows += 1
        if len(bad_examples) < 5:
            bad_examples.append((pid, n, repr(r["Human_Rankings"]), repr(r["LLM_Rankings"]), str(e)))
        continue

    for item in range(1, n + 1):
        target = f"{pid}_resp{item}"
        rows.append({"target": target, "rater": "human", "rating": rank_norm(hr[item], n)})
        rows.append({"target": target, "rater": "llm",   "rating": rank_norm(lr[item], n)})

long_df = pd.DataFrame(rows, columns=["target", "rater", "rating"])

print("Total rows in use:", len(use))
print("Skipped rows (unparseable):", bad_rows)
print("long_df shape:", long_df.shape)
print("Some skipped examples:", bad_examples[:3])

long_df.head(10)


Total rows in use: 195
Skipped rows (unparseable): 0
long_df shape: (1006, 3)
Some skipped examples: []


,target,rater,rating
0,1hv4zdy_resp1,human,0.0
1,1hv4zdy_resp1,llm,1.0
2,1hv4zdy_resp2,human,0.0
3,1hv4zdy_resp2,llm,0.5
4,1hv4zdy_resp3,human,1.0
5,1hv4zdy_resp3,llm,0.0
6,11f4o14_resp1,human,0.0
7,11f4o14_resp1,llm,1.0
8,11f4o14_resp2,human,1.0
9,11f4o14_resp2,llm,0.0
